In [3]:
import sys
!{sys.executable} -m pip install imbalanced-learn

  Using cached imbalanced_learn-0.14.2-py3-none-any.whl.metadata (8.9 kB)
  Using cached sklearn_compat-0.1.6-py3-none-any.whl.metadata (22 kB)
Using cached imbalanced_learn-0.14.2-py3-none-any.whl (236 kB)
Using cached sklearn_compat-0.1.6-py3-none-any.whl (22 kB)

   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]




[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE

In [6]:
df = pd.read_csv('CleandData.csv')
columns_to_drop = ['year', 'claim_amount_usd']
df_cleaned = df.drop(columns=columns_to_drop, errors='ignore')
X = df_cleaned.drop('fraud_label', axis=1)
y = df_cleaned['fraud_label']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
cat_cols = ['country', 'claim_type']
num_cols = [col for col in X_train.columns if col not in cat_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
    ])

In [9]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [10]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_processed, y_train)

In [11]:
joblib.dump(preprocessor, 'preprocessor.pkl')
joblib.dump((X_train_resampled, X_test_processed, y_train_resampled, y_test), 'processed_data.pkl')

['processed_data.pkl']

In [12]:
X_train, X_test, y_train, y_test = joblib.load('processed_data.pkl')
preprocessor = joblib.load('preprocessor.pkl')

In [13]:
print(X_train[:2])

[[-0.56577116 -0.59565901 -0.14385283  1.01102372 -0.97715915 -0.57623299
  -0.33725561 -0.30364432  1.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          1.          0.          0.          0.        ]
 [-1.40205634  0.8945622  -0.35793061 -0.56059675  0.24017577 -0.57623299
  -0.04736631 -0.30364432  0.          0.          1.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          1.          0.        ]]


In [14]:
print(preprocessor)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['policy_duration_months',
                                  'previous_claims_count',
                                  'reporting_delay_days',
                                  'document_completeness_score', 'claimant_age',
                                  'seasonality_flag', 'amount_deviation_zscore',
                                  'duplicate_claim_flag']),
                                ('cat',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['country', 'claim_type'])])
